In [ ]:
# =============================================================================
# Zelle 01 – Setup Phase 3 + Regressionstest canon.py gegen EDA (NB 02 Zelle 17)
# =============================================================================
# Zweck: Baseline A (deterministischer Kern) aus src/ auf dem Snapshot ausführen.
# Regressionstest: Modul canon.kanonisch() muss für alle 10.289 Einträge der
#   übrigen Quellen denselben kanonischen Pfad liefern wie die EDA (nach NFC,
#   da die EDA den Pfad erst beim Abgleich normalisiert hat).
# Datenschutz: Abweichungen nur privat (reports/errors), public nur Zählwerte.
# =============================================================================

import posixpath
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from playlist_agent.canon import kanonisch, nfc
from playlist_agent.viz_config import ARTEFAKT_MUSTER, stil_aktivieren

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
assert PROJECT_ROOT.name == "_MyMSIT_AI-Agent_repair-playlist", PROJECT_ROOT
RAW, INTERIM, GOLD = (PROJECT_ROOT / "data" / p for p in ("raw", "interim", "gold"))
FIG, TAB, ERR = (PROJECT_ROOT / "reports" / p for p in ("figures", "tables", "errors"))
SEED = 42
cfg = yaml.safe_load((PROJECT_ROOT / "config/config.yaml").read_text(encoding="utf-8"))
stil_aktivieren()


def tabelle_speichern(df: pd.DataFrame, name: str, ordner: Path = TAB) -> Path:
    if not ARTEFAKT_MUSTER.match(name):
        raise ValueError(f"Name '{name}' entspricht nicht <NB>_<Zelle>_<slug>.")
    pfad = ordner / f"{name}.csv"
    df.to_csv(pfad, index=False, encoding="utf-8")
    return pfad


# Ordner der Playlist-Quellen auf dem Stick (wie NB 02 Zelle 17)
BASIS = {"musik_playlists": cfg["quellen"]["musik"],
         "mac_playlists": cfg["quellen"]["mac_playlists"],
         "foobar_asus": cfg["laptops"][0]["zielordner"],
         "foobar_medion": cfg["laptops"][1]["zielordner"]}

# --- Regressionstest ------------------------------------------------------------------
df_eda = pd.read_csv(INTERIM / "02_17_playlist_eintraege_kanonisch.csv")
df_eda["modul"] = [kanonisch(e, posixpath.dirname(f"{BASIS[q]}/{d}"))[0]
                   for e, q, d in zip(df_eda["eintrag"], df_eda["quelle"], df_eda["datei"])]
df_eda["gleich"] = df_eda["modul"] == df_eda["kanonisch"].map(nfc)

df_regression = (df_eda.groupby(["quelle", "pfadform"], as_index=False)
                 .agg(eintraege=("gleich", "size"), gleich=("gleich", "sum")))
df_regression["abweichend"] = df_regression["eintraege"] - df_regression["gleich"]
print(df_regression.to_string(index=False))
print(f"\nGesamt: {df_eda['gleich'].sum()} von {len(df_eda)} gleich")
_ = tabelle_speichern(df_regression, "04_01_regression_canon")
df_eda.loc[~df_eda["gleich"]].to_csv(ERR / "04_01_regression_canon_abweichungen.csv",
                                     index=False, encoding="utf-8")       # privat

In [ ]:
# =============================================================================
# Zelle 02 – Regressionstest parsers.serato gegen EDA (NB 02 Zellen 02, 07, 08)
# =============================================================================
# Modul muss auf dem Snapshot exakt die EDA-Ergebnisse liefern:
#   Crates 167 Dateien, 7.915 Einträge, alle vollständig, eine Version;
#   Einträge identisch mit data/interim/02_02_crate_eintraege.csv (Crate, Position, Pfad);
#   Subcrates 203 Dateien, 6.185 Einträge; database V2: 5.478 Tracks, exakt aufgehend.
# =============================================================================

from playlist_agent.parsers.serato import lese_crate, lese_database_v2

# --- Crates ---------------------------------------------------------------------------
zeilen = []
crate_info = []
for f in sorted((RAW / "crates").glob("*.crate")):
    c = lese_crate(f)
    crate_info.append({"version": c.version, "vollstaendig": c.vollstaendig})
    zeilen += [{"crate": f.stem, "position": k, "pfad": p} for k, p in enumerate(c.pfade, start=1)]
df_modul_crates = pd.DataFrame(zeilen)
df_eda_crates = pd.read_csv(INTERIM / "02_02_crate_eintraege.csv")[["crate", "position", "pfad"]]
gleich_crates = df_modul_crates.reset_index(drop=True).equals(df_eda_crates.reset_index(drop=True))

# --- Subcrates ------------------------------------------------------------------------
sub = [lese_crate(f) for f in sorted((RAW / "serato" / "Subcrates").glob("*.crate"))]

# --- database V2 ----------------------------------------------------------------------
tracks, db_ok = lese_database_v2(RAW / "serato" / "database V2")

df_regression_serato = pd.DataFrame([
    {"pruefung": "Crates: Dateien", "modul": len(crate_info), "erwartet": 167},
    {"pruefung": "Crates: Einträge", "modul": len(df_modul_crates), "erwartet": 7915},
    {"pruefung": "Crates: vollständig", "modul": sum(c["vollstaendig"] for c in crate_info), "erwartet": 167},
    {"pruefung": "Crates: Versionen", "modul": len({c["version"] for c in crate_info}), "erwartet": 1},
    {"pruefung": "Crates: Einträge identisch mit EDA", "modul": int(gleich_crates), "erwartet": 1},
    {"pruefung": "Subcrates: Dateien", "modul": len(sub), "erwartet": 203},
    {"pruefung": "Subcrates: Einträge", "modul": sum(len(s.pfade) for s in sub), "erwartet": 6185},
    {"pruefung": "Subcrates: vollständig", "modul": sum(s.vollstaendig for s in sub), "erwartet": 203},
    {"pruefung": "database V2: Tracks", "modul": len(tracks), "erwartet": 5478},
    {"pruefung": "database V2: exakt aufgehend", "modul": int(db_ok), "erwartet": 1},
])
df_regression_serato["ok"] = df_regression_serato["modul"] == df_regression_serato["erwartet"]
print(df_regression_serato.to_string(index=False))
print(f"\n{df_regression_serato['ok'].sum()} von {len(df_regression_serato)} Prüfungen OK")
_ = tabelle_speichern(df_regression_serato, "04_02_regression_serato")

In [ ]:
# =============================================================================
# Zelle 03 – Regressionstest parsers.playlist gegen EDA (NB 02 Zelle 16)
# =============================================================================
# Erwartung: identische Einträge je Datei und Position wie die EDA, AUSSER bei
#   WPL-Einträgen mit XML-Zeichenreferenzen (&amp; &apos; ...) – die EDA hat sie
#   nicht zurückverwandelt (Lücke beim Übertrag entdeckt, 12.11.2026).
# Gemessen: Anzahl abweichender Einträge, Art der Abweichung, und ob abweichende
#   Einträge nach Korrektur im Index treffen (exakt nach NFC).
# =============================================================================

from playlist_agent.parsers.playlist import lese_playlist

PL = RAW / "playlists"
zeilen = []
for quelle_dir in sorted(p for p in PL.iterdir() if p.is_dir()):
    for f in sorted(quelle_dir.rglob("*")):
        if f.is_file() and f.name != ".gitkeep" and not f.name.startswith("._"):
            for e in lese_playlist(f).eintraege:
                zeilen.append({"quelle": quelle_dir.name,
                               "datei": f.relative_to(quelle_dir).as_posix(), "modul": e})
df_modul_pl = pd.DataFrame(zeilen)
df_eda_pl = pd.read_csv(PL.parent.parent / "interim" / "02_16_playlist_eintraege.csv")

assert len(df_modul_pl) == len(df_eda_pl), f"Anzahl abweichend: {len(df_modul_pl)} vs {len(df_eda_pl)}"
df_vergleich_pl = df_eda_pl[["quelle", "datei", "endung", "eintrag"]].assign(modul=df_modul_pl["modul"].to_numpy())
df_vergleich_pl["gleich"] = df_vergleich_pl["eintrag"] == df_vergleich_pl["modul"]

# --- Abweichungen: Ursache und Wirkung -------------------------------------------------
df_index = pd.read_csv(RAW / "index" / "musik_index.csv")
idx_nfc = set(df_index["pfad_relativ"].map(nfc))
abw = df_vergleich_pl[~df_vergleich_pl["gleich"]].copy()
abw["xml_referenz_in_eda"] = abw["eintrag"].str.contains(r"&(?:amp|apos|quot|lt|gt|#\d+);", regex=True)
for spalte in ("eintrag", "modul"):
    abw[f"treffer_{spalte}"] = [kanonisch(e, posixpath.dirname(f"{BASIS[q]}/{d}"))[0] in idx_nfc
                                for e, q, d in zip(abw[spalte], abw["quelle"], abw["datei"])]

df_regression_pl = pd.DataFrame([
    {"pruefung": "Einträge gesamt", "wert": len(df_vergleich_pl)},
    {"pruefung": "gleich wie EDA", "wert": int(df_vergleich_pl["gleich"].sum())},
    {"pruefung": "abweichend", "wert": len(abw)},
    {"pruefung": "abweichend, davon nur WPL", "wert": int((abw["endung"] == ".wpl").sum())},
    {"pruefung": "abweichend, davon XML-Referenz in EDA-Eintrag", "wert": int(abw["xml_referenz_in_eda"].sum())},
    {"pruefung": "Treffer im Index – EDA-Fassung", "wert": int(abw["treffer_eintrag"].sum())},
    {"pruefung": "Treffer im Index – Modul-Fassung", "wert": int(abw["treffer_modul"].sum())},
])
print(df_regression_pl.to_string(index=False))
_ = tabelle_speichern(df_regression_pl, "04_03_regression_playlist")
abw.to_csv(ERR / "04_03_regression_playlist_abweichungen.csv", index=False, encoding="utf-8")   # privat

### Befund Zellen 01–03 – Regressionstests der Module gegen die EDA

| Modul | Prüfung | Ergebnis |
|---|---|---|
| `canon.py` | kanonische Pfade aller 10.289 Einträge übriger Quellen vs. NB 02 Zelle 17 | 10.289 / 10.289 gleich |
| `parsers/serato.py` | Crates (167, 7.915 Einträge, Reihenfolge), Subcrates (203, 6.185), database V2 (5.478 Tracks) | 10 / 10 Prüfungen OK |
| `parsers/playlist.py` | Einträge aller M3U/WPL vs. NB 02 Zelle 16 | 10.182 gleich, 107 abweichend |

**Korrektur eines EDA-Befunds:** Die 107 Abweichungen sind ausschließlich WPL-Einträge mit XML-Zeichenreferenzen (`&amp;`, `&apos;` …), die die EDA nicht zurückverwandelt hat. EDA-Fassung: 0 Treffer im Index; Modul-Fassung: 105 Treffer (98,1 %).
Damit: Musikordner relativ 5.452 / 7.047 (77,4 %) statt 5.347 (75,9 %); Fehltreffer übriger Quellen 3.837 statt 3.942. Gilt ab Phase 3; die EDA-Zahlen in Notebook 02 bleiben als damaliger Stand dokumentiert.

**Lehre:** Ein Regressionstest gegen frühere Ergebnisse bestätigt nicht nur die Übertragung, er kann auch Fehler der früheren Analyse aufdecken; Abweichungen werden erklärt, nicht weggeglättet.

In [ ]:
# =============================================================================
# Zelle 04 – index.scanne() auf dem Stick: Regression + Drift seit Snapshot 07.11.
# =============================================================================
# Nur lesend (P7-Ausnahme: Datei-Metadaten). Verankerung über anchor.verankere()
#   (Workflow 0: Ordner-Signatur primär, Hostname -> Laptop; Seriennummer nur Bestätigung).
# Vergleich je Pfad mit data/raw/index/musik_index.csv (Snapshot 07.11.2026):
#   nur Snapshot (seitdem entfernt/umbenannt) | nur aktuell (neu) |
#   beide, gleiche Größe | beide, andere Größe.
# Neu: leere Ordner (im Datei-Index unsichtbar, NB 02 Zelle 25 / Hinweis C10).
# Datenschutz: Pfade privat (data/interim, reports/errors), public nur Zählwerte.
# =============================================================================

from playlist_agent.anchor import verankere
from playlist_agent.index import leere_ordner, scanne

ANKER = verankere(cfg)
STICK = ANKER.wurzel
print(f"Laptop: {ANKER.laptop['name']} | Laufwerk {ANKER.laufwerk} "
      f"(wie konfiguriert: {ANKER.laufwerk_wie_konfiguriert}) | Seriennummer: nicht geprüft")
df_index_neu = scanne(STICK, cfg["quellen"]["musik"])
leer = leere_ordner(STICK, cfg["quellen"]["musik"])
df_index_neu.to_csv(INTERIM / "04_04_index_aktuell.csv", index=False, encoding="utf-8")   # privat

df_index_alt = pd.read_csv(RAW / "index" / "musik_index.csv")
df_index_alt["pfad_relativ"] = df_index_alt["pfad_relativ"].map(nfc)
v = df_index_alt[["pfad_relativ", "bytes"]].merge(
    df_index_neu[["pfad_relativ", "bytes"]], on="pfad_relativ", how="outer",
    suffixes=("_alt", "_neu"), indicator=True)
v["status"] = np.select(
    [v["_merge"] == "left_only", v["_merge"] == "right_only", v["bytes_alt"] == v["bytes_neu"]],
    ["nur Snapshot", "nur aktuell", "beide, gleiche Größe"], default="beide, andere Größe")

df_drift = v["status"].value_counts().rename_axis("status").reset_index(name="dateien")
df_drift = pd.concat([df_drift, pd.DataFrame([
    {"status": "Dateien Snapshot 07.11.", "dateien": len(df_index_alt)},
    {"status": "Dateien aktuell", "dateien": len(df_index_neu)},
    {"status": "0-Byte-Dateien aktuell", "dateien": int((df_index_neu["bytes"] == 0).sum())},
    {"status": "leere Ordner aktuell", "dateien": len(leer)},
])], ignore_index=True)
print(f"Stick: {STICK}\n")
print(df_drift.to_string(index=False))
_ = tabelle_speichern(df_drift, "04_04_index_drift")
v[v["status"] != "beide, gleiche Größe"].to_csv(ERR / "04_04_index_drift_dateien.csv",
                                                index=False, encoding="utf-8")   # privat
pd.Series(leer, name="ordner").to_csv(ERR / "04_04_leere_ordner.csv", index=False, encoding="utf-8")  # privat

In [ ]:
# =============================================================================
# Zelle 05 – Baseline A: Linter + Resolver auf echten Daten + synthetischen Testfällen
# =============================================================================
# Index: Snapshot (Zelle 04: unverändert) + SHA-256 der Größen-Dubletten (NB 02
#   Zelle 21) + leere Ordner (Zelle 04). Regeln: config.yaml "resolver" (Entscheidungen 12./14.11.).
# Ablauf je Eintrag: Linter (Workflow 3b, Regel "kein Audio", NB 02 Zelle 26)
#   -> bei Befund "ausgeschlossen", sonst Resolver (Workflow 4).
# Echte Daten: Klassen je eindeutigem Crate-Pfad und je Eintrag übriger Quellen.
# Synthetisch (NB 03 Zelle 06): je Typ
#   richtig_sicher | richtig_vorschlag | FALSCH_SICHER (P5!) | falsch_vorschlag |
#   kein_treffer (fehlt/beschädigt). Für T7/T8 ist kein_treffer die RICHTIGE Antwort.
# =============================================================================

from playlist_agent.linter import ist_stream, pruefe_eintrag
from playlist_agent.resolver import Resolver, regeln_aus_config

AUDIO = cfg["index"]["audio_endungen"]
df_hash = pd.read_csv(INTERIM / "02_21_dubletten_hash.csv")[["pfad_relativ", "sha256"]]
df_idx = df_index_alt.merge(df_hash.assign(pfad_relativ=df_hash["pfad_relativ"].map(nfc)),
                            on="pfad_relativ", how="left")
# Resolver-Regeln aus config.yaml (Entscheidungen 12./14.11.2026; Tippfehler -> Abbruch)
REGELN = regeln_aus_config(cfg)
print("Resolver-Regeln:", REGELN)
resolver = Resolver(df_idx, AUDIO, leere_ordner=leer, regeln=REGELN)


def als_zeile(e) -> dict:
    """Resolver-Ergebnis als Tabellenzeile."""
    return {"klasse": e.klasse, "stufe": e.stufe, "pfad_ziel": e.pfad, "n_kand": len(e.kandidaten)}


def bewerte(pfad: str) -> dict:
    """Stream zuerst (unverändert übernehmen), dann Linter (Workflow 3b), dann Resolver (Workflow 4)."""
    if ist_stream(pfad):
        return {"klasse": "stream", "stufe": "webradio", "pfad_ziel": pfad.strip(), "n_kand": 0}
    b = pruefe_eintrag(pfad, AUDIO)
    if b:
        return {"klasse": b.klasse, "stufe": b.untertyp, "pfad_ziel": None, "n_kand": 0}
    return als_zeile(resolver.aufloesen(pfad))


# --- 1. Echte Daten -----------------------------------------------------------------
crate_pfade = pd.Series(df_modul_crates["pfad"].unique(), name="eintrag")
df_res_crates = pd.concat([crate_pfade, pd.DataFrame([bewerte(p) for p in crate_pfade])],
                          axis=1).assign(quelle="crates")
df_res_pl = df_modul_pl.assign(kanonisch=[e.strip() if ist_stream(e) else kanonisch(e, posixpath.dirname(f"{BASIS[q]}/{d}"))[0]
                                          for e, q, d in zip(df_modul_pl["modul"], df_modul_pl["quelle"], df_modul_pl["datei"])])
df_res_pl = pd.concat([df_res_pl, pd.DataFrame([bewerte(k) for k in df_res_pl["kanonisch"]])],
                      axis=1)
df_res_crates.to_csv(INTERIM / "04_05_resolver_crates.csv", index=False, encoding="utf-8")     # privat
df_res_pl.to_csv(INTERIM / "04_05_resolver_playlists.csv", index=False, encoding="utf-8")      # privat

df_echt = (pd.concat([df_res_crates[["quelle", "klasse", "stufe"]], df_res_pl[["quelle", "klasse", "stufe"]]])
           .groupby(["quelle", "klasse", "stufe"]).size().rename("anzahl").reset_index())
print("=== 1: Echte Daten (Crates: eindeutige Pfade; übrige: Einträge) ===")
print(df_echt.pivot_table(index=["klasse", "stufe"], columns="quelle", values="anzahl",
                          aggfunc="sum", fill_value=0, margins=True, margins_name="Summe").to_string())
_ = tabelle_speichern(df_echt, "04_05_resolver_echt")

# --- 2. Synthetische Testfälle (alle mit Audio-Endung -> Linter ohne Wirkung) ------------
df_syn = pd.read_csv(INTERIM / "03_06_synthetisch.csv")
res = [bewerte(kanonisch(e, "")[0]) for e in df_syn["eintrag"]]
df_syn["klasse"] = [r["klasse"] for r in res]
df_syn["pfad_ziel"] = [r["pfad_ziel"] for r in res]


def bewertung(z) -> str:
    if z["erwartung"] == "fehlt":
        return {"sicher": "FALSCH_SICHER", "vorschlag": "falsch_vorschlag"}.get(z["klasse"], "kein_treffer")
    if z["klasse"] in ("fehlt", "album_beschaedigt", "ausgeschlossen"):
        return "kein_treffer"
    richtig = z["pfad_ziel"] == nfc(z["erwartung"])
    if z["klasse"] == "sicher":
        return "richtig_sicher" if richtig else "FALSCH_SICHER"
    return "richtig_vorschlag" if (richtig or pd.isna(z["pfad_ziel"])) else "falsch_vorschlag"


df_syn["bewertung"] = df_syn.apply(bewertung, axis=1)
df_syn_eval = pd.crosstab(df_syn["typ"], df_syn["bewertung"], margins=True, margins_name="Summe")
print("\n=== 2: Synthetische Testfälle (T7/T8: kein_treffer = richtig) ===")
print(df_syn_eval.to_string())
_ = tabelle_speichern(df_syn_eval.reset_index(), "04_05_resolver_synthetisch")
df_syn[df_syn["bewertung"].isin(["FALSCH_SICHER", "falsch_vorschlag"])].to_csv(
    ERR / "04_05_resolver_synthetisch_fehler.csv", index=False, encoding="utf-8")     # privat

# --- 3. Vorab-Messung für den Writer: Zielpfade, die cp1252 nicht darstellen kann ------
ziele = pd.concat([df_res_crates["pfad_ziel"], df_res_pl["pfad_ziel"]]).dropna().unique()


def cp1252_ok(p: str) -> bool:
    try:
        p.encode("cp1252")
        return True
    except UnicodeEncodeError:
        return False


nicht_ok = [p for p in ziele if not cp1252_ok(p)]
print(f"\n=== 3: Zielpfade gesamt {len(ziele)} | nicht in cp1252 darstellbar: {len(nicht_ok)} ===")
zeichen = sorted({c for p in nicht_ok for c in p if not cp1252_ok(c)})
print("Betroffene Zeichen (Codepoints):", [f"U+{ord(c):04X}" for c in zeichen])

# --- 4. Ganzer Musikordner: cp1252-Tauglichkeit + portugiesische Zeichen ------------
alle = df_index_alt["pfad_relativ"].map(nfc)
pt_zeichen = set("ãõçáéíóúâêôàÃÕÇÁÉÍÓÚÂÊÔÀ")
nicht_ok_alle = alle[~alle.map(cp1252_ok)]
mit_pt = alle[alle.map(lambda p: any(c in pt_zeichen for c in p))]
print(f"\n=== 4: Dateien im Musikordner {len(alle)} | nicht in cp1252: {len(nicht_ok_alle)} "
      f"| mit portugiesischen Zeichen: {len(mit_pt)} (davon nicht in cp1252: {int((~mit_pt.map(cp1252_ok)).sum())}) ===")
print("Nicht darstellbare Zeichen (Codepoints):",
      sorted({f"U+{ord(c):04X}" for p in nicht_ok_alle for c in p if not cp1252_ok(c)}))

### Befund Zellen 04–05 – Index-Drift und erster Lauf der Baseline A (Linter + Resolver)

**Index (Zelle 04):** Musikordner seit Snapshot 07.11. unverändert (7.386/7.386 Dateien gleich nach Pfad und Größe); 23 Dateien mit 0 Byte; neu erfasst: 16 leere Ordner.

**Echte Daten (Zelle 05), Regeln Variante a:**
| Klasse | Crates (2.754 Pfade) | übrige Quellen (10.289 Einträge) |
|---|---|---|
| sicher – Pfad | 2.350 | 6.435 |
| sicher – Pfadende (≥ 2 Teile) | 24 | 2.040 |
| Vorschlag – nur Name / mehrfach | 142 / 0 | 1.028 / 2 |
| Album beschädigt (0-Byte-Ordner) | 230 | 509 |
| ausgeschlossen (kein Audio) | 0 | 21 |
| fehlt | 8 | 254 |

Mehrfach-Kandidaten werden bis auf 2 Fälle über ein eindeutiges Pfadende aufgelöst. „Ordner leer" kommt nicht vor: Kein Fehltreffer liegt in einem der 16 leeren Ordner.

**Synthetische Testfälle (2.252):** 0 falsche „sicher", 0 falsche Vorschläge; harte Negativfälle T7/T8 600/600 korrekt (Rule of Three: Rate falscher Zuordnungen < 0,5 %). Recall „sicher": T1/T5/T6 100 %, T2 99 % (3 Vorschläge); T3/T4 bewusst als Vorschlag (nur Dateiname gleich).

**Korrektur an der Ursache:** 17 Einträge, die in der EDA exakt trafen, zeigen auf Nicht-Audio-Dateien (EDA-Nachtrag NB 02 Zelle 26) → Linter-Regel „kein Audio" vor dem Resolver; Meldung als „ausgeschlossen" mit Untertyp statt „fehlt".

**Offen für P5:** Precision der Stufe „Pfadende" (2.064 sichere Zuordnungen) wird mit einer Gold-Stichprobe aus genau dieser Menge gemessen (Entscheidung 12.11.: Blöcke zu 50, Ziel 300, Abbruch beim ersten Fehler).

In [ ]:
# =============================================================================
# Zelle 06 – Dry Run Serato -> foobar (167 Crates x 2 Laptops) + V1 + V3
# =============================================================================
# Ziel: data/processed/04_06_dryrun/<laptop>/ (lokal, NICHT der Stick; P3/P7).
# Inhalt je Playlist: nur Klasse "sicher", Reihenfolge der Crate (P5: Vorschläge
#   erst nach Nutzerbestätigung); übrige Klassen nur im Bericht (P4).
# Format: .m3u8 UTF-8 mit BOM, '#', CRLF, Laufwerk je Laptop (Entscheidung 12.11.).
# V1: jeder geschriebene Pfad ist Audiodatei im Index.
# V3: Zurücklesen mit parsers.playlist + canon ergibt exakt die geschriebenen Pfade.
# Einheiten: [Playlists] = eine Quell-Crate; [Dateien] = geschriebene .m3u8;
#   [Einträge] = eine Zeile einer Playlist (Pfad + Dateiname); derselbe Song in
#   mehreren Playlists zählt mehrfach.
# =============================================================================

from playlist_agent.index import ist_audio
from playlist_agent.parsers.playlist import lese_playlist
from playlist_agent.writer import eindeutige_dateinamen, schreibe_playlist

DRY = PROJECT_ROOT / "data/processed/04_06_dryrun"
audio_pfade = set(df_idx.loc[ist_audio(df_idx, AUDIO), "pfad_relativ"].map(nfc))
erg_je_pfad = df_res_crates.set_index("eintrag")[["klasse", "pfad_ziel"]].to_dict("index")
dateinamen = eindeutige_dateinamen(list(df_modul_crates["crate"].unique()))   # Kollisionsschutz

# --- Schreiben + Verifizieren je Laptop und Crate -------------------------------
zeilen = []
for laptop in cfg["laptops"]:
    lw, ordner = laptop["laufwerk_beobachtet"], DRY / laptop["name"]
    for crate, g in df_modul_crates.groupby("crate", sort=True):
        klassen = [erg_je_pfad[p]["klasse"] for p in g["pfad"]]
        ziele = [erg_je_pfad[p]["pfad_ziel"] for p in g["pfad"] if erg_je_pfad[p]["klasse"] == "sicher"]
        e = schreibe_playlist(ordner, crate, ziele, lw, dateiname=dateinamen[crate])
        zurueck = [kanonisch(x)[0] for x in lese_playlist(e.datei).eintraege]
        zeilen.append({
            "laptop": laptop["name"], "crate": crate, "status": e.status,
            "eintraege": len(g), "geschrieben": len(ziele),
            **{f"n_{k}": klassen.count(k) for k in ("vorschlag", "fehlt", "album_beschaedigt", "ausgeschlossen")},
            "v1_ok": all(nfc(z) in audio_pfade for z in ziele),
            "v3_ok": zurueck == [nfc(z) for z in ziele],
        })
df_dry = pd.DataFrame(zeilen)
df_dry["vollstaendig"] = df_dry["geschrieben"] == df_dry["eintraege"]   # jeder Eintrag "sicher"
df_dry.to_csv(INTERIM / "04_06_dryrun_je_crate.csv", index=False, encoding="utf-8")   # privat

# --- Zusammenfassung je Laptop, Einheit im Spaltennamen ---------------------------
df_dry_sum = (df_dry.groupby("laptop", as_index=False)
              .agg(playlists=("crate", "size"),
                   eintraege=("eintraege", "sum"), geschrieben=("geschrieben", "sum"),
                   vorschlag=("n_vorschlag", "sum"), fehlt=("n_fehlt", "sum"),
                   beschaedigt=("n_album_beschaedigt", "sum"), ausgeschlossen=("n_ausgeschlossen", "sum"),
                   v1_ok=("v1_ok", "sum"), v3_ok=("v3_ok", "sum"),
                   vollstaendig=("vollstaendig", "sum")))
df_dry_sum.insert(2, "dateien", [len(list((DRY / l).glob("*.m3u8"))) for l in df_dry_sum["laptop"]])

# Einheit je Spalte explizit, da Playlist-Zähler und Eintrags-Zähler sonst verwechselbar sind
EINHEIT = {"playlists": "playlists [Playlists]", "dateien": "dateien [Dateien]",
           "eintraege": "eintraege [Einträge]", "geschrieben": "geschrieben [Einträge]",
           "vorschlag": "vorschlag [Einträge]", "fehlt": "fehlt [Einträge]",
           "beschaedigt": "beschaedigt [Einträge]", "ausgeschlossen": "ausgeschlossen [Einträge]",
           "v1_ok": "v1_ok [Playlists]", "v3_ok": "v3_ok [Playlists]",
           "vollstaendig": "vollstaendig [Playlists]"}
df_dry_sum = df_dry_sum.rename(columns=EINHEIT)
print(df_dry_sum.to_string(index=False))
_ = tabelle_speichern(df_dry_sum, "04_06_dryrun_zusammenfassung")

# --- Unvollständige Playlists: Ursache je Crate -----------------------------------
# Laptops sind inhaltsgleich (nur Laufwerk verschieden) -> Auswertung auf einem Laptop
u = df_dry[(df_dry["laptop"] == "asus") & ~df_dry["vollstaendig"]].copy()
u["ursache"] = u.apply(lambda z: " + ".join(k for k, n in [("vorschlag", z["n_vorschlag"]),
                                                          ("fehlt", z["n_fehlt"]),
                                                          ("beschaedigt", z["n_album_beschaedigt"])] if n > 0), axis=1)
print(f"\nUnvollständige Playlists: {len(u)} von {df_dry['crate'].nunique()} [Playlists]")
print(u["ursache"].value_counts().rename("anzahl [Playlists]").to_frame().to_string())
print("\nNur durch Vorschläge unvollständig -> nach Bestätigung vollständig möglich:",
      int((u["ursache"] == "vorschlag").sum()), "[Playlists]")

In [ ]:
# =============================================================================
# Zelle 07 – Nicht sichere Crate-Einträge im Detail (vorschlag / fehlt / beschädigt)
# =============================================================================
# Zweck: Datenverständnis – was steckt hinter 168 / 13 / 509 [Einträge]?
# Eine Zeile je eindeutigem Quellpfad; Zähler, wie oft und in welchen Playlists.
# Spalten:
#   klasse, stufe, grund   – Entscheidung des Resolvers (Zelle 05)
#   eintrag                – Pfad wie in der Crate gespeichert
#   pfad_ziel              – gefundener Kandidat (nur vorschlag)
#   kandidaten             – alle Kandidaten (bei mehreren)
#   null_byte_ordner       – 0-Byte-Datei, die an Stelle des Album-Ordners liegt (nur beschädigt)
#   n_eintraege [Einträge], n_playlists [Playlists], playlists
# Ausgabe privat (echte Pfade, P11): reports/errors/04_07_nicht_sicher_crates.csv
# Abgleich: 0-Byte-Ordner aus Crates vs. Notebook 02 Zelle 25 (dort 10 Album-Ordner)
# =============================================================================

NICHT_SICHER = ["vorschlag", "fehlt", "album_beschaedigt"]

# 0-Byte-Dateien im Index: Kandidaten für zerstörte Album-Ordner (Notebook 02, Zelle 25)
null_byte = set(df_idx.loc[df_idx["bytes"] == 0, "pfad_relativ"].map(nfc))

def null_byte_vorfahr(eintrag):
    """Erster Vorfahr des kanonischen Pfads, der als 0-Byte-Datei existiert."""
    # Crate-Pfade sind bereits relativ zur Laufwerkswurzel (Musik/...) -> kein kanonisch()
    teile_ = nfc(eintrag).replace("\\", "/").split("/")
    for i in range(len(teile_) - 1, 0, -1):
        kandidat = "/".join(teile_[:i])
        if kandidat in null_byte:
            return kandidat
    return ""

# Resolver-Ergebnis je Eintrag mit Crate-Zugehörigkeit verbinden
spalten_res = [c for c in ["eintrag", "klasse", "stufe", "grund", "pfad_ziel", "kandidaten"]
               if c in df_res_crates.columns]
df_ns = (df_modul_crates[["crate", "pfad"]]
         .merge(df_res_crates[spalten_res], left_on="pfad", right_on="eintrag", how="left")
         .query("klasse in @NICHT_SICHER"))

# Verdichten: eine Zeile je eindeutigem Eintrag
df_ns_eindeutig = (df_ns.groupby("eintrag", as_index=False)
                   .agg(**{c: (c, "first") for c in spalten_res if c != "eintrag"},
                        **{"n_eintraege [Einträge]": ("crate", "size"),
                           "n_playlists [Playlists]": ("crate", "nunique"),
                           "playlists": ("crate", lambda s: " | ".join(sorted(set(s))))}))
df_ns_eindeutig["null_byte_ordner"] = [
    null_byte_vorfahr(e) if k == "album_beschaedigt" else ""
    for e, k in zip(df_ns_eindeutig["eintrag"], df_ns_eindeutig["klasse"])]
df_ns_eindeutig = df_ns_eindeutig.sort_values(["klasse", "eintrag"]).reset_index(drop=True)

df_ns_eindeutig.to_csv(PROJECT_ROOT / "reports/errors/04_07_nicht_sicher_crates.csv",
                       index=False, encoding="utf-8-sig")   # utf-8-sig: Excel zeigt Umlaute korrekt

# --- Kontrolle gegen Zelle 06 --------------------------------------------------------
df_ns_kontrolle = (df_ns_eindeutig.groupby("klasse")
                   .agg(**{"eindeutig [Pfade]": ("eintrag", "size"),
                           "summe [Einträge]": ("n_eintraege [Einträge]", "sum")}))
print(df_ns_kontrolle.to_string())

# Beschädigt: Verteilung auf die 0-Byte-Ordner
print("\nBeschädigt je 0-Byte-Ordner [eindeutige Pfade]:")
print(df_ns_eindeutig.query("klasse == 'album_beschaedigt'")["null_byte_ordner"]
      .value_counts().rename("anzahl").to_frame().to_string())

# --- Abgleich: 0-Byte-Ordner aus Crates vs. Notebook 02 Zelle 25 --------------------
df_25 = pd.read_csv(PROJECT_ROOT / "reports/errors/02_25_null_byte_dateien.csv")
print("\nSpalten 02_25:", list(df_25.columns))

ordner_07 = set(df_ns_eindeutig.loc[df_ns_eindeutig["klasse"] == "album_beschaedigt", "null_byte_ordner"])
pfad_spalte = next(c for c in df_25.columns if "pfad" in c.lower())
df_25["_pfad"] = df_25[pfad_spalte].astype(str).map(nfc).str.replace("\\", "/", regex=False)
df_25["in_crates_07"] = df_25["_pfad"].isin(ordner_07)

print(f"\n0-Byte-Dateien in 02_25: {len(df_25)} | davon Vorfahr eines Crate-Eintrags: {int(df_25['in_crates_07'].sum())}")
print(f"In 07 gefunden, aber nicht in 02_25: {sorted(ordner_07 - set(df_25['_pfad']))}")
print("\nAlle Zeilen 02_25 mit Markierung:")
print(df_25.drop(columns="_pfad").to_string(index=False))

In [ ]:
# =============================================================================
# Zelle 08 – Integritätsprüfung (P12): Ausgangs-Manifest + Fehlalarm-Test
# =============================================================================
# Lauf 1: Manifest des echten Sticks (SHA-256 aller Dateien unter Musik/).
#   Erwartete Dauer aus Messung NB02 Z.21 (2,9 GB in 19 s): 128,4 GB -> ca. 14 min;
#   USB-Lesegeschwindigkeit kann abweichen.
# Lauf 2: sofort erneut scannen; Hash-Übernahme bei gleicher Größe+mtime.
#   Kriterium: 0 neu gehashte Dateien, 0 Befunde (keine Fehlalarme).
# Manifest privat (echte Pfade, P11): data/interim/04_08_manifest_stick.csv.
#   Existiert die Datei schon, wird sie geladen statt neu erzeugt (Zelle überschreibt nie).
# Rein lesend (P1/P7).
# =============================================================================

import time
from playlist_agent.index import scanne, sha256_datei
from playlist_agent.integrity import erstelle_manifest, vergleiche_manifeste, zusammenfassung, warnung_noetig

# Stick-Wurzel aus der Verankerung (Zelle 04)
WURZEL = STICK
MANIFEST = INTERIM / "04_08_manifest_stick.csv"

def hash_mit_fortschritt(datei, _z={"n": 0, "t0": time.perf_counter()}):
    """sha256_datei mit Fortschrittsanzeige alle 500 Dateien."""
    _z["n"] += 1
    if _z["n"] % 500 == 0:
        print(f"  {_z['n']} Dateien gehasht, {time.perf_counter() - _z['t0']:.0f} s")
    return sha256_datei(datei)

# --- Lauf 1: Ausgangs-Manifest ------------------------------------------------------
if MANIFEST.exists():
    df_manifest = pd.read_csv(MANIFEST, encoding="utf-8")
    print(f"Manifest geladen: {len(df_manifest)} [Dateien] (kein Neu-Hash)")
else:
    t0 = time.perf_counter()
    df_manifest, n1 = erstelle_manifest(scanne(WURZEL), WURZEL, hash_fn=hash_mit_fortschritt)
    df_manifest.to_csv(MANIFEST, index=False, encoding="utf-8")
    print(f"Lauf 1: {len(df_manifest)} [Dateien], neu gehasht {n1} [Dateien], "
          f"{(time.perf_counter() - t0) / 60:.1f} min")

# --- Lauf 2: Fehlalarm-Test ---------------------------------------------------------
t0 = time.perf_counter()
df_manifest_2, n2 = erstelle_manifest(scanne(WURZEL), WURZEL, df_alt=df_manifest)
df_befunde = vergleiche_manifeste(df_manifest, df_manifest_2)
print(f"Lauf 2: neu gehasht {n2} [Dateien], {time.perf_counter() - t0:.1f} s")
print(zusammenfassung(df_befunde).to_string(index=False))
print("Warnung nötig:", warnung_noetig(df_befunde))

In [ ]:
# --- Dubletten nach Inhalt (SHA-256): Herkunft der Differenz Dateien vs. eindeutige Hashes ---
def dubletten_kategorie(z):
    if z["bytes"] == 0:
        return "0 Byte"
    if z["appledouble"]:
        return "AppleDouble"
    return "Audio" if z["endung"] in AUDIO else f"sonstige ({z['endung']})"

df_dup = df_manifest.assign(kategorie=df_manifest.apply(dubletten_kategorie, axis=1),
                            in_gruppe=df_manifest.duplicated("sha256", keep=False),
                            ueberzaehlig=df_manifest.duplicated("sha256", keep="first"))
df_dup_sum = (df_dup[df_dup["in_gruppe"]].groupby("kategorie")
              .agg(**{"dateien in Gruppen [Dateien]": ("sha256", "size"),
                      "gruppen [Hashes]": ("sha256", "nunique"),
                      "ueberzaehlig [Dateien]": ("ueberzaehlig", "sum")})
              .sort_values("ueberzaehlig [Dateien]", ascending=False))
print(f"Dateien {len(df_manifest)} | eindeutige Hashes {df_manifest['sha256'].nunique()} | "
      f"überzählig {int(df_dup['ueberzaehlig'].sum())} [Dateien]")
print(df_dup_sum.to_string())
_ = tabelle_speichern(df_dup_sum.reset_index(), "04_08_dubletten_nach_inhalt")

### Ergebnis Zelle 08 – Integritätsprüfung (P12)

**Fehlalarm-Test auf echten Daten bestanden**
- Lauf 1 (Ausgangs-Manifest): 7.386 [Dateien] gehasht in 14,0 min (Vorhersage aus NB02 Z.21: ca. 14 min).
- Lauf 2 (sofort erneut): 0 [Dateien] neu gehasht, 4,6 s; 0 [Befunde] in allen 6 Typen; keine Warnung.
- Hash-Übernahme bei gleicher Größe+mtime funktioniert: Folgeläufe kosten Sekunden statt Minuten.

**Dubletten nach Inhalt** (`reports/tables/04_08_dubletten_nach_inhalt.csv`)
- 7.386 [Dateien], 6.633 eindeutige Hashes -> 753 überzählige [Dateien], vollständig aufgeschlüsselt.
- Audio: 45 Gruppen à 2 = exakt die 45 bitgleichen Paare aus NB02 Z.21 -> Vorfilter über Größe hat keine Dublette übersehen.
- Übrige 708 [Dateien]: .jpg 373, AppleDouble 244, .ini 25, 0 Byte 22, .ifo/.bup 13, .m3u 12, .url 10, sonstige 9 – keine Musik.
- Hypothesen (nicht geprüft): .jpg = in mehrere Album-Ordner kopierte Cover; .bup = DVD-Sicherungskopie von .ifo.

**Bedeutung:** Die Integritätsprüfung vergleicht je Pfad; inhaltsgleiche Dateien an verschiedenen Pfaden sind kein Befund.
Das Manifest `data/interim/04_08_manifest_stick.csv` (privat) ist die Referenz für den nächsten Lauf.

In [ ]:
# =============================================================================
# Zelle 09 – Pflichtbericht (P4) aus Dry Run (Z.06), Resolver (Z.05/07), Integrität (Z.08)
# =============================================================================
# Ein Bericht je Ziel-Laptop. Enthält echte Pfade -> nur reports/run_reports/ (privat, P11).
# Ausgabe im Notebook nur Abschnitte 1–2 (keine Pfade).
# =============================================================================

from datetime import datetime
from playlist_agent.report import erstelle_bericht, speichere_bericht

RUN_REPORTS = PROJECT_ROOT / "reports/run_reports"

# Alle nicht sicheren Crate-Einträge (inkl. ausgeschlossen) mit 0-Byte-Ordner
df_bericht_eintraege = (df_modul_crates[["crate", "pfad"]]
                        .merge(df_res_crates[spalten_res], left_on="pfad", right_on="eintrag", how="left")
                        .query("klasse != 'sicher'")
                        .rename(columns={"crate": "playlist"}))
df_bericht_eintraege["null_byte_ordner"] = [
    null_byte_vorfahr(e) if k == "album_beschaedigt" else ""
    for e, k in zip(df_bericht_eintraege["eintrag"], df_bericht_eintraege["klasse"])]

jetzt = datetime.now()
for laptop in cfg["laptops"]:
    kopf = {"zeit": jetzt.strftime("%Y-%m-%d %H:%M:%S"),
            "laptop": laptop["name"],
            "laufwerk": laptop["laufwerk_beobachtet"],
            "quelle": "Serato-Crates",
            "modus": "Dry Run (lokal, data/processed/04_06_dryrun)",
            "manifest": MANIFEST.name}
    df_bericht_pl = df_dry[df_dry["laptop"] == laptop["name"]].rename(columns={"crate": "playlist"})
    text = erstelle_bericht(kopf, df_bericht_pl, df_bericht_eintraege, df_befunde)
    datei = speichere_bericht(text, RUN_REPORTS / laptop["name"], jetzt)
    print(f"{laptop['name']}: {datei.relative_to(PROJECT_ROOT)}  ({len(text.splitlines())} Zeilen)")

# Abschnitte 1–2 des letzten Berichts zeigen (enthalten keine Pfade)
print("\n" + text.split("\n## 3")[0])

### Ergebnis Zelle 09 – Pflichtbericht (P4)

- Je Ziel-Laptop ein Bericht in `reports/run_reports/<laptop>/lauf_<zeit>.md` (privat, echte Pfade; nie überschrieben).
- Reihenfolge: 1 Integrität (P12) vor allen Playlist-Ergebnissen, da Datenverlust Vorrang hat.
- Inhalte stimmen mit Zellen 06–08 überein: 167 [Playlists], 128 vollständig, 39 unvollständig, V1/V3 0 Fehler;
  7.915 [Einträge] = 7.225 sicher + 168 Vorschlag + 13 fehlt + 509 beschädigt + 0 ausgeschlossen.
- Umfang 260 Zeilen; davon 200 Tabellenzeilen = 142 Vorschläge + 8 fehlt + 11 Ordner mit 0 Byte + 39 unvollständige Playlists.
- Beschädigte Einträge werden je Ordner mit 0 Byte verdichtet (11 statt 230 Zeilen) -> direkt als Suchliste für eine Sicherung nutzbar.

In [ ]:
# =============================================================================
# Zelle 10 – Prüf-Playlist aller Vorschläge (Abschnitt 4) + Gold-Label-Vorlage
# =============================================================================
# Zweck: Gold-Prüfung aller 142 eindeutigen Vorschläge durch den Nutzer in foobar.
# Playlist lokal (nicht auf dem Stick, P3/P7): data/processed/04_10_pruefung/
# Vorlage privat (P11): data/gold/04_10_vorschlag_labels.csv (Trenner ';' für Excel)
#   Spalte label: r = richtiger Song, f = falscher Song, ? = unklar
# Reihenfolge Playlist = Reihenfolge Vorlage (Spalte nr).
# =============================================================================

from playlist_agent.writer import schreibe_playlist

PRUEF_ORDNER = PROJECT_ROOT / "data/processed/04_10_pruefung"
GOLD_VORLAGE = PROJECT_ROOT / "data/gold/04_10_vorschlag_labels.csv"
LW = str(WURZEL)[:2]                      # Laufwerk dieses Laptops, z. B. "D:"

df_vorschlag = (df_ns_eindeutig.query("klasse == 'vorschlag'")
                .sort_values("eintrag", ignore_index=True))
df_vorschlag.insert(0, "nr", range(1, len(df_vorschlag) + 1))
assert df_vorschlag["pfad_ziel"].map(nfc).isin(audio_pfade).all(), "Kandidat nicht im Index (V1)"

e = schreibe_playlist(PRUEF_ORDNER, "04_10_pruefung_vorschlaege", list(df_vorschlag["pfad_ziel"]), LW)
print(f"Playlist: {e.datei.relative_to(PROJECT_ROOT)} | {e.eintraege} [Einträge] | Status {e.status}")

if GOLD_VORLAGE.exists():
    print(f"Vorlage existiert bereits, nicht überschrieben: {GOLD_VORLAGE.relative_to(PROJECT_ROOT)}")
else:
    (df_vorschlag[["nr", "eintrag", "pfad_ziel", "n_playlists [Playlists]"]]
     .rename(columns={"eintrag": "original_in_crate", "pfad_ziel": "kandidat"})
     .assign(label="", kommentar="")
     .to_csv(GOLD_VORLAGE, sep=";", index=False, encoding="utf-8-sig"))
    print(f"Vorlage: {GOLD_VORLAGE.relative_to(PROJECT_ROOT)} | {len(df_vorschlag)} [Zeilen]")

In [ ]:
# =============================================================================
# Zelle 11 – Gold-Labels Vorschläge: Sammelprüfung in foobar (Nutzerangabe 14.11.2026)
# =============================================================================
# Quelle: Prüf-Playlist Zelle 10, alle 142 eindeutigen Vorschläge in foobar geprüft.
# Ergebnis Nutzer: alle richtig -> label "r" für alle Zeilen.
# Eingabe:  data/gold/04_10_vorschlag_labels.csv (Vorlage, unverändert)
# Ausgabe:  data/gold/04_11_vorschlag_labels.csv (privat, P11; nie überschrieben)
# Entscheidung 14.11.2026 (Regel a): Regel "Name eindeutig" bleibt Klasse "vorschlag";
#   bestätigte Einträge werden als Nutzerentscheidung geschrieben (P5 bleibt gewahrt).
# =============================================================================

GOLD_LABELS = PROJECT_ROOT / "data/gold/04_11_vorschlag_labels.csv"

df_gold_vorschlag = pd.read_csv(GOLD_VORLAGE, sep=";", encoding="utf-8-sig")
assert len(df_gold_vorschlag) == 142 and df_gold_vorschlag["label"].isna().all(), "Vorlage unerwartet"

df_gold_vorschlag["label"] = "r"
df_gold_vorschlag["kommentar"] = "Sammelprüfung foobar, Nutzerangabe 14.11.2026"

if GOLD_LABELS.exists():
    print(f"Existiert bereits, nicht überschrieben: {GOLD_LABELS.relative_to(PROJECT_ROOT)}")
else:
    df_gold_vorschlag.to_csv(GOLD_LABELS, sep=";", index=False, encoding="utf-8-sig")
    print(f"Gespeichert: {GOLD_LABELS.relative_to(PROJECT_ROOT)}")

n, n_r = len(df_gold_vorschlag), int((df_gold_vorschlag["label"] == "r").sum())
print(f"Labels: {n} [Vorschläge], richtig {n_r}, falsch 0 | "
      f"Precision {n_r / n:.1%}, obere 95-%-Grenze Fehlerrate {1 - 0.05 ** (1 / n):.2%} (Clopper-Pearson, 0 Fehler)")

In [ ]:
# =============================================================================
# Zelle 12 – Vorschläge je Ordnerpaar (alter Ordner -> neuer Ordner)
# =============================================================================
# Zweck: Reparaturart (Playlist anpassen vs. Datei zurückverschieben) ist eine
#   Nutzerentscheidung; Verschiebungen erfolgen meist ordnerweise -> Entscheidung je
#   Ordnerpaar statt je Track. Identität ist geklärt (Z.11: 142/142 richtig).
# alter_ordner_existiert: Ordner der Crate existiert noch (Einzeldatei verschoben)
#   oder nicht (ganzer Ordner verschoben/umbenannt).
# Ausgabe privat (echte Pfade, P11): reports/errors/04_12_vorschlaege_je_ordnerpaar.csv
# =============================================================================

import posixpath
from playlist_agent.integrity import ordner_aus_pfaden

ordner_stick = ordner_aus_pfaden(df_manifest["pfad_relativ"])

df_paar = df_gold_vorschlag.assign(
    alter_ordner=df_gold_vorschlag["original_in_crate"].map(lambda p: posixpath.dirname(nfc(p).replace("\\", "/"))),
    neuer_ordner=df_gold_vorschlag["kandidat"].map(lambda p: posixpath.dirname(nfc(p))))
df_paar_sum = (df_paar.groupby(["alter_ordner", "neuer_ordner"], as_index=False)
               .agg(**{"tracks [Pfade]": ("nr", "size"),
                       "eintraege [Einträge]": ("n_playlists [Playlists]", "sum")})
               .sort_values("tracks [Pfade]", ascending=False, ignore_index=True))
df_paar_sum["alter_ordner_existiert"] = df_paar_sum["alter_ordner"].isin(ordner_stick)
df_paar_sum.to_csv(PROJECT_ROOT / "reports/errors/04_12_vorschlaege_je_ordnerpaar.csv",
                   index=False, encoding="utf-8-sig")

print(f"Vorschläge: {len(df_paar)} [Pfade] -> {len(df_paar_sum)} [Ordnerpaare]")
print("Tracks je Ordnerpaar:", df_paar_sum["tracks [Pfade]"].describe()[["min", "50%", "max"]].to_dict())
print(df_paar_sum.groupby("alter_ordner_existiert")
      .agg(**{"ordnerpaare": ("tracks [Pfade]", "size"), "tracks [Pfade]": ("tracks [Pfade]", "sum")})
      .to_string())

# --- Ordnerpaare zur Nutzerentscheidung (Nummern O1..O7 für die Antwort) ---------------
df_paar_sum.insert(0, "id", [f"O{i}" for i in range(1, len(df_paar_sum) + 1)])
for z in df_paar_sum.itertuples(index=False):
    print(f"{z.id}: {z[3]} Tracks, {z[4]} Einträge | alter Ordner existiert: {z.alter_ordner_existiert}")
    print(f"     alt: {z.alter_ordner}")
    print(f"     neu: {z.neuer_ordner}")

In [ ]:
# =============================================================================
# Zelle 13 – Pflichtbericht mit Entscheidungen über Vorschläge (P13)
# =============================================================================
# Entscheidungsprotokoll: data/decisions/entscheidungen.csv (privat, P11; laufend gepflegt).
#   Fehlt die Datei -> noch keine Entscheidung -> alle Vorschläge offen ("?").
# loese_auf: Track > Ordnerpaar > offen. Bericht Abschnitt 4 je Ordnerpaar, 4b Verschiebe-Vorschläge.
# reload: report.py wurde nach dem Import geändert (editable install, Modul im Kernel gecacht).
# =============================================================================

import importlib
import playlist_agent.decisions, playlist_agent.report
importlib.reload(playlist_agent.decisions)
importlib.reload(playlist_agent.report)
from playlist_agent.decisions import lade_entscheidungen, loese_auf
from playlist_agent.report import erstelle_bericht, speichere_bericht

ENTSCHEIDUNGEN = PROJECT_ROOT / "data/decisions/entscheidungen.csv"
df_protokoll = lade_entscheidungen(ENTSCHEIDUNGEN)
print(f"Entscheidungsprotokoll: {len(df_protokoll)} [Zeilen]")

# Aktion je Vorschlag auflösen und an die Bericht-Einträge (Z.09) anhängen
ist_v = df_bericht_eintraege["klasse"] == "vorschlag"
df_aufg = loese_auf(df_bericht_eintraege.loc[ist_v, ["eintrag", "pfad_ziel"]].drop_duplicates(), df_protokoll)
df_bericht_e2 = df_bericht_eintraege.merge(df_aufg[["eintrag", "pfad_ziel", "aktion", "entschieden_auf"]],
                                           on=["eintrag", "pfad_ziel"], how="left")
print("Vorschläge nach Aktion [Einträge]:",
      df_bericht_e2.loc[df_bericht_e2["klasse"] == "vorschlag", "aktion"].value_counts().to_dict())

jetzt = datetime.now()
for laptop in cfg["laptops"]:
    kopf = {"zeit": jetzt.strftime("%Y-%m-%d %H:%M:%S"), "laptop": laptop["name"],
            "laufwerk": laptop["laufwerk_beobachtet"], "quelle": "Serato-Crates",
            "modus": "Dry Run (lokal, data/processed/04_06_dryrun)", "manifest": MANIFEST.name,
            "entscheidungen": f"{len(df_protokoll)} Protokollzeilen"}
    df_bericht_pl = df_dry[df_dry["laptop"] == laptop["name"]].rename(columns={"crate": "playlist"})
    text = erstelle_bericht(kopf, df_bericht_pl, df_bericht_e2, df_befunde)
    datei = speichere_bericht(text, RUN_REPORTS / laptop["name"], jetzt)
    print(f"{laptop['name']}: {datei.relative_to(PROJECT_ROOT)}  ({len(text.splitlines())} Zeilen, "
          f"{text.count(chr(10) + '### O')} [Ordnerpaare])")

### Ergebnis Zellen 10–13 – Vorschläge: Identität geprüft, Reparaturart als Nutzerentscheidung (P13)

**Identität (Gold, Z.10/11):** Prüf-Playlist aller 142 eindeutigen Vorschläge, in foobar vom Nutzer geprüft:
142/142 richtiger Song. Precision 100 %, obere 95-%-Grenze Fehlerrate 2,09 % (Clopper-Pearson, 0 Fehler).
-> Regel "Name eindeutig, anderer Ordner" bleibt Klasse "vorschlag" (Regel a, 14.11.2026): 2,09 % belegt keine 100 % (P5);
   außerdem 1.028 ungeprüfte Name-Vorschläge in den übrigen Quellen.

**Reparaturart (Z.12):** Gold-Label belegt nur die Identität, nicht ob der neue Ort gewollt ist.
- 142 Vorschläge -> 7 Ordnerpaare (alter -> neuer Ordner); 4 Paare / 70 Tracks: alter Ordner existiert nicht mehr,
  3 Paare / 72 Tracks: alter Ordner existiert noch.
- Muster: Sammelordner, Umbenennung, Unterordner, fremder Set-Ordner, Fremdpfad.
- Leitfrage: deterministische Gruppierung reduziert 142 Entscheidungen auf 7 (-95 %); Track-Entscheidung bleibt immer möglich.

**Entscheidungslogik (Z.13, `decisions.py`):** a = Playlist anpassen | b = Zurückverschieben vorschlagen (nur Bericht, P1) | ? = offen.
Vorrang Track > Ordnerpaar > offen; Protokoll nur anhängend, jüngste Zeile gilt; anderer Kandidat entwertet die Track-Entscheidung.
Stand 15.11.2026: 0 Entscheidungen -> 168 [Einträge] offen; Bericht zeigt 7 Ordnerpaare, Abschnitt 4b leer.
Die Software ändert die Datenbasis nicht; die Entscheidungen trifft der Nutzer später über Bericht, Agent (Phase 5) und UI (Phase 7).

In [ ]:
# =============================================================================
# Zelle 14 – Gold-Stichprobe Regel "pfadende -> sicher" (P5): Prüf-Playlist + Vorlage
# =============================================================================
# Grundgesamtheit: eindeutige Paare (Eintrag kanonisch -> Ziel) mit Stufe "pfadende",
#   Crates + übrige Quellen. Obergrenze je Zielordner (Entscheidung 12.11.: Klumpen
#   vermeiden), dann Zufallsstichprobe n = 300 (Rule of Three: 0 Fehler -> < 1 %),
#   zufällige Reihenfolge (Entscheidung Phase 2: Prüfung in Blöcken, Abbruch beim 1. Fehler).
# Playlist lokal (nicht auf dem Stick, P3/P7); Vorlage privat (P11).
# =============================================================================

SEED_GOLD, N_GOLD, MAX_JE_ORDNER = 20261115, 300, 5
GOLD_PE = PROJECT_ROOT / "data/gold/04_14_pfadende_vorlage.csv"

df_pe = pd.concat([
    df_res_crates.loc[df_res_crates["stufe"] == "pfadende", ["eintrag", "pfad_ziel", "quelle"]],
    df_res_pl.loc[df_res_pl["stufe"] == "pfadende", ["kanonisch", "pfad_ziel", "quelle"]]
        .rename(columns={"kanonisch": "eintrag"}),
]).drop_duplicates(["eintrag", "pfad_ziel"], ignore_index=True)
df_pe["zielordner"] = df_pe["pfad_ziel"].map(posixpath.dirname)
print(f"Grundgesamtheit pfadende: {len(df_pe)} [Paare] in {df_pe['zielordner'].nunique()} [Zielordner]")

df_pe_begr = df_pe.groupby("zielordner").sample(frac=1, random_state=SEED_GOLD).groupby("zielordner").head(MAX_JE_ORDNER)
n = min(N_GOLD, len(df_pe_begr))
df_pe_s = df_pe_begr.sample(n=n, random_state=SEED_GOLD).reset_index(drop=True)
df_pe_s.insert(0, "nr", range(1, n + 1))
print(f"Nach Obergrenze {MAX_JE_ORDNER}/Ordner: {len(df_pe_begr)} [Paare] -> Stichprobe {n} [Paare]")
print("Stichprobe je Quelle [Paare]:", df_pe_s["quelle"].value_counts().to_dict())

e = schreibe_playlist(PRUEF_ORDNER, "04_14_pruefung_pfadende", list(df_pe_s["pfad_ziel"]), LW)
print(f"Playlist: {e.datei.relative_to(PROJECT_ROOT)} | {e.eintraege} [Einträge] | Status {e.status}")
if GOLD_PE.exists():
    print(f"Vorlage existiert bereits, nicht überschrieben: {GOLD_PE.relative_to(PROJECT_ROOT)}")
else:
    (df_pe_s[["nr", "quelle", "eintrag", "pfad_ziel"]]
     .rename(columns={"eintrag": "original", "pfad_ziel": "kandidat"})
     .assign(label="", kommentar="")
     .to_csv(GOLD_PE, sep=";", index=False, encoding="utf-8-sig"))
    print(f"Vorlage: {GOLD_PE.relative_to(PROJECT_ROOT)} | {n} [Zeilen]")

In [ ]:
# =============================================================================
# Zelle 15 – Gold-Labels Regel "pfadende -> sicher": Prüfung in foobar (Nutzerangabe 15.11.2026)
# =============================================================================
# Ergebnis Nutzer: alle Pfade der Stichprobe führen zum richtigen Titel -> label "r".
# Belegt nur die Identität; Reparaturart (Pfad/Titelname ändern) bleibt Nutzerentscheidung (P13).
# Eingabe:  data/gold/04_14_pfadende_vorlage.csv (unverändert)
# Ausgabe:  data/gold/04_15_pfadende_labels.csv (privat, P11; nie überschrieben)
# =============================================================================

GOLD_PE_LABELS = PROJECT_ROOT / "data/gold/04_15_pfadende_labels.csv"

df_gold_pe = pd.read_csv(GOLD_PE, sep=";", encoding="utf-8-sig")
assert df_gold_pe["label"].isna().all(), "Vorlage enthält bereits Labels"
df_gold_pe["label"] = "r"
df_gold_pe["kommentar"] = "Prüfung foobar, Nutzerangabe 15.11.2026"

if GOLD_PE_LABELS.exists():
    print(f"Existiert bereits, nicht überschrieben: {GOLD_PE_LABELS.relative_to(PROJECT_ROOT)}")
else:
    df_gold_pe.to_csv(GOLD_PE_LABELS, sep=";", index=False, encoding="utf-8-sig")
    print(f"Gespeichert: {GOLD_PE_LABELS.relative_to(PROJECT_ROOT)}")

n = len(df_gold_pe)
print(f"Labels pfadende: {n} [Paare], richtig {n}, falsch 0 | je Quelle: {df_gold_pe['quelle'].value_counts().to_dict()}")
print(f"Precision 100 %, obere 95-%-Grenze Fehlerrate {1 - 0.05 ** (1 / n):.2%} (Clopper-Pearson, 0 Fehler)")

### Ergebnis Zellen 14–15 – Gold-Prüfung Regel "pfadende -> sicher" (P5)

- Grundgesamtheit: 825 eindeutige Paare (Eintrag -> Ziel) in 112 Zielordnern (2.064 [Einträge] mit Wiederholungen).
- Stichprobe: Obergrenze 5 je Zielordner (423 Paare), dann Zufall n = 300, zufällige Reihenfolge, Seed 20261115.
  Quellen: Musikordner 288, Crates 8, Mac 4.
- Prüfung in foobar (Nutzer, 15.11.2026): 300/300 führen zum richtigen Titel.
  Obere 95-%-Grenze Fehlerrate 0,99 % (Clopper-Pearson, 0 Fehler) -> Ziel < 1 % (Phase 2) erreicht; Regel bleibt "sicher".
- Einschränkung: Schicht Crates nur 8 Paare (allein < 31 %); Grenze gilt für die Grundgesamtheit nach Obergrenze je Ordner.
- Das Label belegt nur die Identität. Ob Pfad oder Titelname geändert werden sollen, entscheidet der Nutzer (P13);
  die Datenbasis bleibt bis Projektende unverändert (Entscheidung 15.11.2026).

In [ ]:
# =============================================================================
# Zelle 16 – Dry Run übrige Quellen -> foobar (Mac-M3U, .m3u/.wpl Musikordner, foobar) + V1 + V3
# =============================================================================
# Ziel: data/processed/04_16_dryrun_quellen/<laptop>/<quelle>/ (lokal, P3/P7).
#   Je Quelle eigener Unterordner: Namensregeln zwischen Quellen (_mac, Sync) = Phase 6.
# Playlist-Name = relativer Pfad der Quelldatei ohne Endung, "/" -> " - " (eindeutig je Quelle).
# Geschrieben: Klasse "sicher" + Vorschläge mit Aktion a (P13, Protokoll Z.13); Reihenfolge der Quelle.
# Einheiten: [Playlists] = eine Quelldatei; [Einträge] = eine Zeile einer Playlist.
# =============================================================================

DRY_Q = PROJECT_ROOT / "data/processed/04_16_dryrun_quellen"

# Aktion je Vorschlag aus dem Protokoll (derzeit leer -> alle offen)
df_v_pl = df_res_pl.loc[df_res_pl["klasse"] == "vorschlag", ["kanonisch", "pfad_ziel"]].drop_duplicates()
akt_pl = (loese_auf(df_v_pl.rename(columns={"kanonisch": "eintrag"}), df_protokoll)
          .set_index(["eintrag", "pfad_ziel"])["aktion"].to_dict())

def ziel_quelle(z):
    """Zu schreibender Pfad oder None (sicher -> Ziel; Vorschlag mit Aktion a -> Kandidat)."""
    if z.klasse in ("sicher", "stream"):      # Stream: URL unverändert (Entscheidung 15.11.2026)
        return z.pfad_ziel
    if z.klasse == "vorschlag" and akt_pl.get((z.kanonisch, z.pfad_ziel)) == "a":
        return z.pfad_ziel
    return None

zeilen = []
for laptop in cfg["laptops"]:
    lw = laptop["laufwerk_beobachtet"]
    for quelle, gq in df_res_pl.groupby("quelle", sort=True):
        namen = {d: str(Path(d).with_suffix("")).replace("\\", "/").replace("/", " - ") for d in gq["datei"].unique()}
        dateinamen_q = eindeutige_dateinamen(list(namen.values()))
        for datei, g in gq.groupby("datei", sort=True):          # Zeilenreihenfolge in g = Quelle
            ziele = [z for z in map(ziel_quelle, g.itertuples(index=False)) if z is not None]
            e = schreibe_playlist(DRY_Q / laptop["name"] / quelle, namen[datei], ziele, lw,
                                  dateiname=dateinamen_q[namen[datei]])
            zurueck = [x if ist_stream(x) else kanonisch(x)[0] for x in lese_playlist(e.datei).eintraege]
            kl = list(g["klasse"])
            zeilen.append({
                "laptop": laptop["name"], "quelle": quelle, "datei": datei, "status": e.status,
                "eintraege": len(g), "geschrieben": len(ziele),
                **{f"n_{k}": kl.count(k) for k in ("vorschlag", "fehlt", "album_beschaedigt", "ausgeschlossen", "stream")},
                "v1_ok": all(nfc(z) in audio_pfade for z in ziele if not ist_stream(z)),
                "v3_ok": zurueck == [nfc(z) for z in ziele],
            })
df_dry_q = pd.DataFrame(zeilen)
# vollständig = alle Audio-Einträge geschrieben; ausgeschlossene (kein Audio) fehlen nicht (Befund 15.11.2026)
df_dry_q["vollstaendig"] = df_dry_q["geschrieben"] == df_dry_q["eintraege"] - df_dry_q["n_ausgeschlossen"]
df_dry_q["leer"] = df_dry_q["geschrieben"] == 0
df_dry_q.to_csv(INTERIM / "04_16_dryrun_quellen_je_playlist.csv", index=False, encoding="utf-8")   # privat

df_dry_q_sum = (df_dry_q[df_dry_q["laptop"] == "asus"].groupby("quelle")
                .agg(**{"playlists [Playlists]": ("datei", "size"),
                        "eintraege [Einträge]": ("eintraege", "sum"),
                        "geschrieben [Einträge]": ("geschrieben", "sum"),
                        "vorschlag [Einträge]": ("n_vorschlag", "sum"),
                        "fehlt [Einträge]": ("n_fehlt", "sum"),
                        "beschaedigt [Einträge]": ("n_album_beschaedigt", "sum"),
                        "ausgeschlossen [Einträge]": ("n_ausgeschlossen", "sum"),
                        "stream [Einträge]": ("n_stream", "sum"),
                        "v1_ok [Playlists]": ("v1_ok", "sum"), "v3_ok [Playlists]": ("v3_ok", "sum"),
                        "vollstaendig [Playlists]": ("vollstaendig", "sum"),
                        "leer [Playlists]": ("leer", "sum")}))
df_dry_q_sum.loc["Summe"] = df_dry_q_sum.sum()
print("Je Quelle (Laptop asus; medion inhaltsgleich, nur Laufwerk anders):")
print(df_dry_q_sum.to_string())
print("\nLaptops inhaltsgleich (Zähler):",
      df_dry_q[df_dry_q["laptop"] == "asus"][["eintraege", "geschrieben"]].sum().tolist()
      == df_dry_q[df_dry_q["laptop"] == "medion"][["eintraege", "geschrieben"]].sum().tolist())
_ = tabelle_speichern(df_dry_q_sum.reset_index(), "04_16_dryrun_quellen_zusammenfassung")

In [ ]:
# --- Unvollständige und leere Playlists: Ursache je Playlist (Laptop asus) ---------------
u_q = df_dry_q[(df_dry_q["laptop"] == "asus") & ~df_dry_q["vollstaendig"]].copy()
u_q["ursache"] = u_q.apply(lambda z: " + ".join(k for k, n in [("vorschlag", z["n_vorschlag"]),
                                                              ("fehlt", z["n_fehlt"]),
                                                              ("beschaedigt", z["n_album_beschaedigt"]),
                                                              ("ausgeschlossen", z["n_ausgeschlossen"])] if n > 0), axis=1)
print(f"\nUnvollständig: {len(u_q)} [Playlists], davon leer {int(u_q['leer'].sum())}")
print(u_q.groupby(["quelle", "leer"])["ursache"].value_counts()
      .rename("anzahl [Playlists]").to_frame().to_string())
print("\nNur durch Vorschläge unvollständig (nach Entscheidung a vollständig möglich):",
      int((u_q["ursache"] == "vorschlag").sum()), "[Playlists]")

In [ ]:
# --- Unvollständige Playlists: Ursache je Playlist + je Ursache ein Beispiel (Laptop asus) ---
u_q = df_dry_q[(df_dry_q["laptop"] == "asus") & ~df_dry_q["vollstaendig"]].copy()
u_q["ursache"] = u_q.apply(lambda z: " + ".join(k for k, n in [("vorschlag", z["n_vorschlag"]),
                                                              ("fehlt", z["n_fehlt"]),
                                                              ("beschaedigt", z["n_album_beschaedigt"]),
                                                              ("ausgeschlossen", z["n_ausgeschlossen"])] if n > 0), axis=1)
print(f"\nUnvollständig: {len(u_q)} [Playlists], davon leer {int(u_q['leer'].sum())}")
print(u_q.groupby(["quelle", "leer"])["ursache"].value_counts()
      .rename("anzahl [Playlists]").to_frame().to_string())

print("\n=== Je Ursache ein Beispiel (erste Playlist alphabetisch) ===")
for (quelle, leer_, ursache), g in u_q.sort_values("datei").groupby(["quelle", "leer", "ursache"], sort=True):
    z = g.iloc[0]
    print(f"\n[{quelle} | leer={leer_} | {ursache}] {len(g)} [Playlists] mit dieser Ursache")
    print(f"  Quelldatei: {STICK / BASIS[quelle] / z['datei']}")
    print(f"  {z['eintraege']} [Einträge], geschrieben {z['geschrieben']}")
    zeilen_pl = df_res_pl[(df_res_pl["quelle"] == quelle) & (df_res_pl["datei"] == z["datei"])].head(8)
    for r in zeilen_pl.itertuples(index=False):
        print(f"    {r.klasse:<17} {r.stufe:<16} | {r.modul}")
        if pd.notna(r.pfad_ziel) and r.klasse != "sicher":
            print(f"    {'':<17} {'-> Kandidat':<16} | {r.pfad_ziel}")

In [ ]:
# --- Unvollständige Playlists: Ursache je Playlist + je Ursache ein Beispiel (Laptop asus) ---
u_q = df_dry_q[(df_dry_q["laptop"] == "asus") & ~df_dry_q["vollstaendig"]].copy()
u_q["ursache"] = u_q.apply(lambda z: " + ".join(k for k, n in [("vorschlag", z["n_vorschlag"]),
                                                              ("fehlt", z["n_fehlt"]),
                                                              ("beschaedigt", z["n_album_beschaedigt"]),
                                                              ("ausgeschlossen", z["n_ausgeschlossen"])] if n > 0), axis=1)
print(f"\nUnvollständig: {len(u_q)} [Playlists], davon leer {int(u_q['leer'].sum())}")
print(u_q.groupby(["quelle", "leer"])["ursache"].value_counts()
      .rename("anzahl [Playlists]").to_frame().to_string())

print("\n=== Je Ursache ein Beispiel (erste Playlist alphabetisch) ===")
for (quelle, leer_, ursache), g in u_q.sort_values("datei").groupby(["quelle", "leer", "ursache"], sort=True):
    z = g.iloc[0]
    print(f"\n[{quelle} | leer={leer_} | {ursache}] {len(g)} [Playlists] mit dieser Ursache")
    print(f"  Quelldatei: {STICK / BASIS[quelle] / z['datei']}")
    print(f"  {z['eintraege']} [Einträge], geschrieben {z['geschrieben']}")
    zeilen_pl = df_res_pl[(df_res_pl["quelle"] == quelle) & (df_res_pl["datei"] == z["datei"])].head(8)
    for r in zeilen_pl.itertuples(index=False):
        print(f"    {r.klasse:<17} {r.stufe:<16} | {r.modul}")
        if pd.notna(r.pfad_ziel) and r.klasse != "sicher":
            print(f"    {'':<17} {'-> Kandidat':<16} | {r.pfad_ziel}")

In [ ]:
# --- Transparenz: ALLE nicht sicheren Einträge einer Playlist (Position, Original, Kandidat) ---
def zeige_nicht_sicher(quelle: str, datei_teil: str) -> None:
    """Alle Einträge mit Klasse != sicher; Position = Zeile in der Quell-Playlist (1-basiert)."""
    g = df_res_pl[(df_res_pl["quelle"] == quelle)
                  & df_res_pl["datei"].str.contains(datei_teil, case=False, regex=False)]
    for datei, gd in g.groupby("datei"):
        gd = gd.reset_index(drop=True)
        ns = gd[gd["klasse"] != "sicher"]
        print(f"\n[{quelle}] {datei}: {len(gd)} [Einträge], nicht sicher {len(ns)}")
        for pos, r in ns.iterrows():
            print(f"  #{pos + 1:<4} {r['klasse']:<17} {r['stufe']:<16} | {r['modul']}")
            if pd.notna(r["pfad_ziel"]):
                print(f"  {'':<5} {'':<17} {'-> Kandidat':<16} | {r['pfad_ziel']}")

# Aufruf interaktiv: zeige_nicht_sicher(quelle, teil_des_dateinamens).
# Keine echten Datei-/Ordnernamen im Code (P11) -> Beispiele über die Daten auswählen, nicht über Namen:
for quelle_, datei_ in u_q.sort_values("datei")[["quelle", "datei"]].head(3).itertuples(index=False):
    zeige_nicht_sicher(quelle_, datei_)

In [ ]:
# =============================================================================
# Zelle 17 – Messung Parser-Fehler A ('..' bleibt in kanonischen Pfaden) und B (%XX ohne file://)
# =============================================================================
# Befund Nutzerprüfung 15.11.2026: A = 0-Byte-Ordner nicht erkannt (Sunrise Event Mamaia),
#   B = AnlageE-Einträge '%20' nicht dekodiert. Nur Messung, keine Änderung.
# Risiko B: echte Dateinamen mit %XX-Muster würden durch Dekodieren verfälscht -> Index prüfen.
# =============================================================================

import re
from urllib.parse import unquote
from playlist_agent.canon import pfadform

PCT = re.compile(r"%[0-9A-Fa-f]{2}")
DOTDOT = re.compile(r"(?:^|/)\.\.(?:/|$)")

# --- A: '..' im kanonischen Pfad ------------------------------------------------------
ist_a = df_res_pl["kanonisch"].str.contains(DOTDOT)
print(f"A: '..' im kanonischen Pfad: {int(ist_a.sum())} [Einträge] in {df_res_pl.loc[ist_a, 'datei'].nunique()} [Playlists]")
print("   Pfadform:", df_res_pl.loc[ist_a, "modul"].map(pfadform).value_counts().to_dict())
neu_a = [bewerte(posixpath.normpath(k))["klasse"] for k in df_res_pl.loc[ist_a, "kanonisch"]]
print("   Klasse vorher:", df_res_pl.loc[ist_a, "klasse"].value_counts().to_dict())
print("   Klasse nach normpath:", pd.Series(neu_a).value_counts().to_dict())

# --- B: %XX ohne file:// ----------------------------------------------------------------
ist_b = df_res_pl["modul"].str.contains(PCT) & (df_res_pl["modul"].map(pfadform) != "url")
print(f"\nB: %XX ohne file://: {int(ist_b.sum())} [Einträge] in {df_res_pl.loc[ist_b, 'datei'].nunique()} [Playlists]")
neu_b = [bewerte(posixpath.normpath(kanonisch(unquote(e), posixpath.dirname(f"{BASIS[q]}/{d}"))[0]))["klasse"]
         for e, q, d in zip(df_res_pl.loc[ist_b, "modul"], df_res_pl.loc[ist_b, "quelle"], df_res_pl.loc[ist_b, "datei"])]
print("   Klasse vorher:", df_res_pl.loc[ist_b, "klasse"].value_counts().to_dict())
print("   Klasse nach Dekodierung:", pd.Series(neu_b).value_counts().to_dict())

# --- Risiko-Gegenprobe im Index ---------------------------------------------------------
print(f"\nIndex: Dateinamen mit %XX-Muster: {int(df_index_alt['pfad_relativ'].str.contains(PCT).sum())} [Dateien]"
      f" | Pfade mit '..'-Segment: {int(df_index_alt['pfad_relativ'].str.contains(DOTDOT).sum())} [Dateien]")

In [ ]:
# =============================================================================
# Zelle 18 – Gold-Abdeckung Regel "pfadende -> sicher" nach Fix A (Zelle 17)
# =============================================================================
# Befund: Fix A ('..' auflösen) macht viele frühere pfadende-Fälle zu exakten Pfaden.
# Frage: Wie viele der 300 Gold-Paare (Z.14/15) fallen heute noch unter "pfadende"?
#   Nur diese belegen die heutige Regel; Grenze nach Clopper-Pearson neu berechnen.
# Vergleich über kanonischen Eintrag nach normpath (Gold-Originale enthalten teils '..').
# =============================================================================

df_pe_jetzt = pd.concat([
    df_res_crates.loc[df_res_crates["stufe"] == "pfadende", ["eintrag", "pfad_ziel"]],
    df_res_pl.loc[df_res_pl["stufe"] == "pfadende", ["kanonisch", "pfad_ziel"]].rename(columns={"kanonisch": "eintrag"}),
]).drop_duplicates(ignore_index=True)
paare_jetzt = set(zip(df_pe_jetzt["eintrag"].map(nfc), df_pe_jetzt["pfad_ziel"].map(nfc)))

gold_paare = list(zip(df_gold_pe["original"].map(lambda p: nfc(posixpath.normpath(p))), df_gold_pe["kandidat"].map(nfc)))
n_gold_jetzt = sum(p in paare_jetzt for p in gold_paare)

print(f"pfadende heute: {len(paare_jetzt)} [Paare] (vorher 825)")
print(f"Gold-Paare (300) heute noch pfadende: {n_gold_jetzt} [Paare], heute exakter Pfad o. a.: {300 - n_gold_jetzt}")
if n_gold_jetzt:
    print(f"Obere 95-%-Grenze Fehlerrate für die heutige Regel: {1 - 0.05 ** (1 / n_gold_jetzt):.2%} (0 Fehler)")

In [ ]:
# =============================================================================
# Zelle 19 – Vollprüfung Regel "pfadende -> sicher": Prüf-Playlist der ungeprüften Paare
# =============================================================================
# Nach Fix A: 138 pfadende-Paare, davon 57 bereits geprüft (Z.15, Z.18) -> Rest 81.
# Ziel: Vollerhebung des heutigen Bestands (138/138). Reihenfolge zufällig (Seed).
# Playlist lokal (P3/P7), Vorlage privat (P11).
# =============================================================================

GOLD_PE_REST = PROJECT_ROOT / "data/gold/04_19_pfadende_rest_vorlage.csv"

gepr = set(gold_paare)
df_pe_rest = df_pe_jetzt[[ (nfc(e), nfc(z)) not in gepr
                           for e, z in zip(df_pe_jetzt["eintrag"], df_pe_jetzt["pfad_ziel"])]]
df_pe_rest = df_pe_rest.sample(frac=1, random_state=SEED_GOLD).reset_index(drop=True)
df_pe_rest.insert(0, "nr", range(1, len(df_pe_rest) + 1))
print(f"Ungeprüfte pfadende-Paare: {len(df_pe_rest)} [Paare]")

e = schreibe_playlist(PRUEF_ORDNER, "04_19_pruefung_pfadende_rest", list(df_pe_rest["pfad_ziel"]), LW)
print(f"Playlist: {e.datei.relative_to(PROJECT_ROOT)} | {e.eintraege} [Einträge] | Status {e.status}")
if GOLD_PE_REST.exists():
    print(f"Vorlage existiert bereits, nicht überschrieben: {GOLD_PE_REST.relative_to(PROJECT_ROOT)}")
else:
    (df_pe_rest.rename(columns={"eintrag": "original", "pfad_ziel": "kandidat"})
     .assign(label="", kommentar="")
     .to_csv(GOLD_PE_REST, sep=";", index=False, encoding="utf-8-sig"))
    print(f"Vorlage: {GOLD_PE_REST.relative_to(PROJECT_ROOT)} | {len(df_pe_rest)} [Zeilen]")

In [ ]:
# =============================================================================
# Zelle 20 – Gold-Labels Restprüfung "pfadende" (Nutzerangabe 15.11.2026) + Vollerhebung
# =============================================================================
# Ergebnis Nutzer: alle 81 ungeprüften Paare (Z.19) führen zum richtigen Titel -> label "r".
# Zusammen mit 57 Paaren aus Z.15 (heute noch pfadende, Z.18): Vollerhebung 138/138.
# Belegt nur die Identität; Reparaturart bleibt Nutzerentscheidung (P13).
# Eingabe:  data/gold/04_19_pfadende_rest_vorlage.csv (unverändert)
# Ausgabe:  data/gold/04_20_pfadende_rest_labels.csv (privat, P11; nie überschrieben)
# =============================================================================

GOLD_PE_REST_LABELS = PROJECT_ROOT / "data/gold/04_20_pfadende_rest_labels.csv"

df_gold_pe_rest = pd.read_csv(GOLD_PE_REST, sep=";", encoding="utf-8-sig")
assert df_gold_pe_rest["label"].isna().all(), "Vorlage enthält bereits Labels"
df_gold_pe_rest["label"] = "r"
df_gold_pe_rest["kommentar"] = "Prüfung foobar, Nutzerangabe 15.11.2026"

if GOLD_PE_REST_LABELS.exists():
    print(f"Existiert bereits, nicht überschrieben: {GOLD_PE_REST_LABELS.relative_to(PROJECT_ROOT)}")
else:
    df_gold_pe_rest.to_csv(GOLD_PE_REST_LABELS, sep=";", index=False, encoding="utf-8-sig")
    print(f"Gespeichert: {GOLD_PE_REST_LABELS.relative_to(PROJECT_ROOT)}")

n_rest = len(df_gold_pe_rest)
n_voll = n_gold_jetzt + n_rest
print(f"Restprüfung: {n_rest} [Paare] richtig | Vollerhebung pfadende: {n_voll} von {len(paare_jetzt)} [Paare] richtig")
print(f"Obere 95-%-Grenze Fehlerrate (für künftige Daten, 0 Fehler in {n_voll}): {1 - 0.05 ** (1 / n_voll):.2%}")

### Ergebnis Zellen 16–20 – Übrige Quellen, zwei Parser-Fehler, Streams, Vollprüfung "pfadende"

**Dry Run übrige Quellen (Z.16):** 185 [Playlists] (foobar 5 + 7, Mac 29, Musikordner 144), 10.289 [Einträge];
geschrieben 8.495, Vorschlag 1.026, fehlt 71, beschädigt 678, ausgeschlossen 19, Stream 2; V1/V3 185/185;
vollständig 57, leer 10. Je Quelle eigener Unterordner; Namensregeln zwischen Quellen = Phase 6.

**Nutzerprüfung (Z.16/17, Gold):** Detailansicht `zeige_nicht_sicher` (alle nicht sicheren Einträge mit Position)
deckte auf:
- Fehler A (belegt): '..' in absoluten Windows-Pfaden nicht aufgelöst -> 2.191 Einträge betroffen;
  Fix in `canon.py` (normpath für alle Formen): fehlt 171 -> 2, beschädigt +169, sicher +4.
- Fehler B (belegt): %XX ohne file:// nicht dekodiert -> 14 Einträge; Fix als Resolver-Rückfall nur bei "fehlt",
  da 1 echte Datei ein %XX-Muster im Namen trägt: fehlt 14 -> sicher 14.
- Kennzahl "vollständig" zählte Nicht-Audio als fehlend -> neu: alle Audio-Einträge geschrieben.
- Webradio-Streams: Entscheidung 15.11.2026 -> Klasse "stream", unverändert geschrieben (2 Einträge).
- Phase-4-Fälle (Gold): Zeichen in Quell-Playlist verloren (3), Datei umbenannt (2) -> Fuzzy/ML.
- Phase-6-Hinweis: viele Playlists liegen doppelt als .m3u und .wpl mit gleichem Inhalt.

**Folge für P5 (Z.18–20):** Nach Fix A fiel "pfadende" von 825 auf 138 eindeutige Paare (die Regel hatte
überwiegend Fehler A ausgeglichen). Nur 57 der 300 Gold-Paare fielen noch darunter (Grenze 5,12 %).
Vollprüfung der restlichen 81 durch den Nutzer: 138/138 richtig -> Vollerhebung des heutigen Bestands;
für künftige Daten obere 95-%-Grenze 2,15 %.

**Lehre:** Eine Regel, die "sicher" behauptet, kann einen vorgelagerten Fehler verdecken. Gold-Belege gelten
nur für den Stand, auf dem sie erhoben wurden; nach jeder Änderung der Kaskade Abdeckung neu messen.

In [ ]:
# =============================================================================
# Zelle 21 – Messung für create_from_folder: Tracknummer vs. natürliche Sortierung des Dateinamens
# =============================================================================
# Frage: Taugt die natürliche Sortierung ('2 x' vor '10 x') als Ersatzregel, wenn Tracknummern fehlen?
# Beleg: in Ordnern MIT vollständigen, eindeutigen Tracknummern beide Reihenfolgen vergleichen.
# Tags aus NB 02 (tinytag); Datei und Spaltennamen werden ermittelt und ausgegeben.
# Einheit [Ordner] = Ordner mit >= 2 Audiodateien (direkt darin, ohne Unterordner).
# =============================================================================

tag_dateien = sorted(INTERIM.glob("02_*tag*.csv"))
print("Tag-Dateien:", [p.name for p in tag_dateien])
df_tags = pd.read_csv(tag_dateien[-1])
spalte_track = next(c for c in df_tags.columns if "track" in c.lower() and "total" not in c.lower())
print("Spalten:", list(df_tags.columns), "| Tracknummer-Spalte:", spalte_track)

def track_int(v):
    """'3', '3/12', 3.0 -> 3; leer/ungültig -> None."""
    m = re.match(r"\s*(\d+)", str(v)) if pd.notna(v) else None
    return int(m.group(1)) if m else None

def natur_schluessel(name: str):
    """Natürliche Sortierung: Zahlen als Zahl, Text ohne Groß/Klein."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", name)]

df_t = df_tags.assign(pfad_relativ=df_tags["pfad_relativ"].map(nfc),
                      ordner=lambda d: d["pfad_relativ"].map(posixpath.dirname),
                      name=lambda d: d["pfad_relativ"].map(posixpath.basename),
                      track=lambda d: d[spalte_track].map(track_int))
zeilen = []
for ordner, g in df_t.groupby("ordner"):
    if len(g) < 2:
        continue
    alle = g["track"].notna().all()
    eindeutig = alle and g["track"].is_unique
    gleich = None
    if eindeutig:
        gleich = list(g.sort_values("track")["name"]) == sorted(g["name"], key=natur_schluessel)
    zeilen.append({"ordner": ordner, "dateien": len(g), "track_alle": alle,
                   "track_eindeutig": eindeutig, "natur_gleich_track": gleich})
df_ord = pd.DataFrame(zeilen)
df_ord.to_csv(ERR / "04_21_ordner_reihenfolge.csv", index=False, encoding="utf-8")   # privat

n = len(df_ord)
n_eind = int(df_ord["track_eindeutig"].sum())
n_gleich = int(df_ord["natur_gleich_track"].fillna(False).sum())
print(f"\nOrdner mit >= 2 Audiodateien: {n} [Ordner]")
print(f"  Tracknummer bei allen Dateien: {int(df_ord['track_alle'].sum())} | davon eindeutig: {n_eind}")
print(f"  Tracknummer fehlt bei mind. 1 Datei: {n - int(df_ord['track_alle'].sum())}")
print(f"  Natürliche Sortierung = Track-Reihenfolge: {n_gleich} von {n_eind} [Ordner]"
      + (f" ({n_gleich / n_eind:.1%}, obere/untere Grenze folgt nach Befund)" if n_eind else ""))

In [ ]:
# =============================================================================
# Zelle 22 – create_from_folder: Ordnertyp A/S/E und Reihenfolge (Entscheidung Nutzer 15.11.2026)
# =============================================================================
# Typ S Set-Mitschnitt: alle Dateinamen = gemeinsamer Anfang + Zahl am Ende (beliebig viele Ziffern,
#        Trenner Leerzeichen/_/-) und Zahlen eindeutig -> Reihenfolge nach Endzahl.
#        Vorrang vor Tags (Nummer vom Nutzer vergeben).
# Typ A Album: Tracknummern vollständig + eindeutig -> nach Tracknummer.
# Typ E Einzeltitel: sonst -> natürliche Sortierung (keine echte Sequenz).
# Messung: Anzahl je Typ; Typ der 9 Abweichungen aus Z.21; Konflikte S vs. Tags.
# =============================================================================

ENDZAHL = re.compile(r"^(.*?)[\s_\-]*(\d+)$")

def set_reihenfolge(namen: list[str]) -> list[str] | None:
    """Reihenfolge nach Endzahl, wenn alle Namen gemeinsamen Anfang + eindeutige Endzahl haben; sonst None."""
    teile_ = [ENDZAHL.match(posixpath.splitext(n)[0]) for n in namen]
    if not all(teile_):
        return None
    anfang = {m.group(1).strip().lower() for m in teile_}
    zahlen = [int(m.group(2)) for m in teile_]
    if len(anfang) != 1 or len(set(zahlen)) != len(zahlen):
        return None
    return [n for _, n in sorted(zip(zahlen, namen))]

zeilen = []
for ordner, g in df_t.groupby("ordner"):
    if len(g) < 2:
        continue
    namen = list(g["name"])
    s = set_reihenfolge(namen)
    track_ok = g["track"].notna().all() and g["track"].is_unique
    nach_track = list(g.sort_values("track")["name"]) if track_ok else None
    typ = "S" if s else ("A" if track_ok else "E")
    zeilen.append({"ordner": ordner, "dateien": len(g), "typ": typ,
                   "track_ok": track_ok, "konflikt_s_track": bool(s and track_ok and s != nach_track)})
df_typ = pd.DataFrame(zeilen).merge(df_ord[["ordner", "natur_gleich_track"]], on="ordner", how="left")
df_typ.to_csv(ERR / "04_22_ordnertyp.csv", index=False, encoding="utf-8")   # privat

print("Ordner je Typ [Ordner]:", df_typ["typ"].value_counts().to_dict())
print("Typ S mit vollständigen Tags:", int((df_typ["typ"].eq("S") & df_typ["track_ok"]).sum()),
      "| davon Konflikt Endzahl vs. Tags:", int(df_typ["konflikt_s_track"].sum()))
print("Typ der 9 Abweichungen (Z.21):", df_typ.loc[df_typ["natur_gleich_track"] == False, "typ"].value_counts().to_dict())  # noqa: E712
_ = tabelle_speichern(df_typ.groupby("typ").agg(**{"ordner [Ordner]": ("ordner", "size"),
                                                   "dateien [Dateien]": ("dateien", "sum")}).reset_index(),
                      "04_22_ordnertyp_zusammenfassung")

print("\n=== Je Typ 2 Beispiele (erste 3 Titel in der gewählten Reihenfolge) ===")
for typ, g in df_typ.groupby("typ"):
    for ordner in g["ordner"].head(2):
        gg = df_t[df_t["ordner"] == ordner]
        namen = list(gg["name"])
        reihe = (set_reihenfolge(namen) if typ == "S" else
                 list(gg.sort_values("track")["name"]) if typ == "A" else
                 sorted(namen, key=natur_schluessel))
        print(f"[{typ}] {ordner}: {reihe[:3]}")

In [ ]:
# =============================================================================
# Zelle 23 – create_from_folder im Dry Run: alle Ordner mit >= 2 Audiodateien
# =============================================================================
# Modul ordner.py (Typ S Endzahl > A Tracknummer > E Dateiname, Entscheidung 15.11.2026).
# Ziel: data/processed/04_23_dryrun_ordner/<laptop>/ (lokal, P3/P7); Name = Ordnerpfad, "/" -> " - ".
# Regression: Regel je Ordner muss Typ aus Zelle 22 entsprechen (S=endzahl, A=tracknummer, E=dateiname).
# V1: jeder Pfad im Audio-Index; V3: Zurücklesen = geschriebene Pfade.
# =============================================================================

from playlist_agent.ordner import playlist_aus_ordner

DRY_O = PROJECT_ROOT / "data/processed/04_23_dryrun_ordner"
audio_liste = list(df_t["pfad_relativ"])
tracks = {p: int(t) for p, t in zip(df_t["pfad_relativ"], df_t["track"]) if pd.notna(t)}   # int, nicht float
REGEL_ZU_TYP = {"endzahl": "S", "tracknummer": "A", "dateiname": "E"}

namen_o = {o: o.replace("/", " - ") for o in df_typ["ordner"]}
dateinamen_o = eindeutige_dateinamen(list(namen_o.values()))
zeilen = []
for laptop in cfg["laptops"]:
    lw = laptop["laufwerk_beobachtet"]
    for ordner in df_typ["ordner"]:
        pfade, regel = playlist_aus_ordner(audio_liste, ordner, tracks)
        e = schreibe_playlist(DRY_O / laptop["name"], namen_o[ordner], pfade, lw,
                              dateiname=dateinamen_o[namen_o[ordner]])
        zurueck = [kanonisch(x)[0] for x in lese_playlist(e.datei).eintraege]
        zeilen.append({"laptop": laptop["name"], "ordner": ordner, "regel": regel, "titel": len(pfade),
                       "v1_ok": all(nfc(p) in audio_pfade for p in pfade), "v3_ok": zurueck == pfade})
df_dry_o = pd.DataFrame(zeilen).merge(df_typ[["ordner", "typ"]], on="ordner")
df_dry_o["regel_wie_typ"] = df_dry_o["regel"].map(REGEL_ZU_TYP) == df_dry_o["typ"]
df_dry_o.to_csv(INTERIM / "04_23_dryrun_ordner.csv", index=False, encoding="utf-8")   # privat

a = df_dry_o[df_dry_o["laptop"] == "asus"]
df_dry_o_sum = (a.groupby("regel").agg(**{"playlists [Playlists]": ("ordner", "size"),
                                          "titel [Einträge]": ("titel", "sum")}).reset_index())
print(df_dry_o_sum.to_string(index=False))
print(f"\nRegel = Typ aus Z.22: {int(a['regel_wie_typ'].sum())} von {len(a)} | "
      f"V1 ok: {int(a['v1_ok'].sum())} | V3 ok: {int(a['v3_ok'].sum())} [Playlists]")
_ = tabelle_speichern(df_dry_o_sum, "04_23_dryrun_ordner_zusammenfassung")

## Abschluss Notebook 04 – Phase 3: Deterministischer Kern (Baseline A)

### Ergebnis in einem Satz
Der deterministische Kern übersetzt alle Quellen in foobar-Playlists für beide Laptops, ohne einen einzigen
falschen "sicheren" Treffer in allen Gold-Prüfungen; was er nicht auflöst, ist überwiegend **Datenverlust**
(0-Byte-Ordner) oder eine **Nutzerentscheidung** (Vorschläge), nicht fehlende Intelligenz.

### Module (src/playlist_agent, 132 Tests grün)
canon · parsers (Serato, M3U/WPL) · index · linter (kein Audio, Stream) · resolver (Kaskade, Regeln aus config.yaml,
%-Rückfall) · writer (.m3u8, Backup, Stream-Durchreichung) · integrity (P12) · report (P4) · decisions (P13) ·
anchor (Workflow 0) · ordner (create_from_folder) · privacy_check (Git-Hook) · CI (pytest, Windows)

### Kennzahlen (Dry Run, je Laptop; [Einträge] = Zeilen einer Playlist)
| Quelle | Playlists | Einträge | sicher | Vorschlag | fehlt | beschädigt | ausgeschl. | Stream |
|---|---|---|---|---|---|---|---|---|
| Serato-Crates | 167 | 7.915 | 7.225 | 168 | 13 | 509 | 0 | 0 |
| übrige Quellen | 185 | 10.289 | 8.493 | 1.026 | 71 | 678 | 19 | 2 |
| Ordner (create_from_folder) | 311 | 5.113 | 5.113 | – | – | – | – | – |
V1 (Datei existiert) und V3 (Round-Trip) für alle 663 Playlists ok; 0 Fehlalarme der Integritätsprüfung auf dem echten Stick.

### Belege für P5 (Precision "sicher")
| Regel | Prüfung | Ergebnis | obere 95-%-Grenze Fehlerrate |
|---|---|---|---|
| pfadende -> sicher | Vollerhebung heutiger Bestand (Z.18–20) | 138/138 | 2,15 % (künftige Daten) |
| Name eindeutig -> Vorschlag | alle 142 Crate-Vorschläge (Z.10/11) | 142/142 | 2,09 % -> bleibt Vorschlag |
| synthetische Hard Negatives T7/T8 | 600 Fälle | 0 falsch sicher | 0,5 % |

### Entscheidungen in Phase 3
immer .m3u8 (100 Dateinamen nicht cp1252) · Linter "kein Audio" · Regel a (Name -> Vorschlag) · P12 Integrität ·
P13 Identität getrennt von Reparaturart, Entscheidung je Track (optional je Ordnerpaar) · Datenbasis bis Projektende
unverändert · Webradio-Streams übernehmen · Reihenfolge Ordner: Endzahl > Tracknummer > Dateiname (Typ S/A/E,
automatisch je Aufruf) · Resolver-Regeln in config.yaml · Privacy-Check als Git-Hook, PRIVATE_TERMS in .env

### Gefundene und behobene Fehler (durch Nutzerprüfung = Gold)
- WPL-XML-Referenzen (EDA-Fehler, 107 Einträge) · Dateinamen-Kollision im Writer · ".." in absoluten Pfaden
  (2.191 Einträge; "pfadende" hatte den Fehler verdeckt) · %XX ohne file:// (14) · Kennzahl "vollständig" ·
  Privacy-Check brach an ';'-Gold-Dateien ab, Dateinamen mit Leerzeichen wären ungeprüft geblieben.
- Lehre: Gold-Belege gelten nur für den Stand, auf dem sie erhoben wurden -> nach jeder Änderung der Kaskade neu messen.

### Leitfrage – Zwischenstand
Deterministisch gelöst: exakter Pfad, Pfadende, Kodierungen, Streams, Reihenfolge, Gruppierung von 142 Vorschlägen
auf 7 Ordnerpaare (-95 % Entscheidungen). **Raum für Phase 4 (Fuzzy/ML):** 71 + 13 "fehlt"-Einträge, davon belegt
umbenannte Dateien und verlorene Zeichen; 2 Mehrfach-Kandidaten. **Nicht lösbar ohne Sicherung:** 11 bzw. 23
0-Byte-Ordner (509 + 678 Einträge).

### Übergabe an Phase 4–6
Fuzzy/ML für "fehlt" (Gold: 5 Fälle aus Z.16) · Sync und Varianten (_mac, _2021, .m3u/.wpl-Dubletten) · Agent-Werkzeuge:
zeige_nicht_sicher, Bericht, Entscheidungen.